# 04 - Bonus Task: Document Intelligence Option A

This notebook covers **Bonus Task 4, Option A: Working Extraction Prototype with API Comparison**.

It shows:

1. Extraction of Total Revenue 2024, Net Income 2024, and Total Assets 2024 from the English PDF.
2. The same extraction attempt on the Arabic PDF.
3. Accuracy comparison against the ground-truth values from the case-study data dictionary.
4. Where Arabic extraction is fragile even when it succeeds on this sample.
5. Comparison of local Tesseract OCR with commercial OCR APIs.
6. A pilot-stage and scale-stage recommendation with cost estimates.


## 1. Setup

In [1]:
from pathlib import Path
import json
import subprocess
import sys

import pandas as pd
from IPython.display import Markdown, display

pd.set_option('display.max_columns', 160)
pd.set_option('display.max_colwidth', 160)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'scripts').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

REPORT_DIR = PROJECT_ROOT / 'reports' / 'document_intelligence'
SCRIPT_PATH = PROJECT_ROOT / 'scripts' / 'run_document_intelligence.py'

print('Project root:', PROJECT_ROOT)
print('Python:', sys.executable)
print('Bonus script:', SCRIPT_PATH)


Project root: /Users/adi/Documents/ai_credit_intelligence
Python: /Users/adi/anaconda3/bin/python
Bonus script: /Users/adi/Documents/ai_credit_intelligence/scripts/run_document_intelligence.py


In [2]:
def run_project_script(script_path):
    result = subprocess.run(
        [sys.executable, str(script_path)],
        cwd=PROJECT_ROOT,
        check=True,
        text=True,
        capture_output=True,
    )
    print(result.stdout)
    if result.stderr:
        print(result.stderr)


def load_json(path):
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)


## 2. Run the Extraction Prototype

The script renders each PDF page with ImageMagick, OCRs the images with Tesseract, then applies label-anchored extraction for the three required 2024 values.

For Arabic, the script uses `script/Arabic` when available and applies digit/OCR-confusion normalization because Arabic table numbers are often misread by local OCR.


In [3]:
run_project_script(SCRIPT_PATH)

results = load_json(REPORT_DIR / 'extraction_results.json')
extraction_df = pd.read_csv(REPORT_DIR / 'extraction_results.csv')

extraction_df


Document intelligence bonus task complete.
English accuracy: {'target_fields': 3, 'fields_extracted': 3, 'fields_matched_ground_truth': 3, 'field_match_rate': 1.0}
Arabic accuracy: {'target_fields': 3, 'fields_extracted': 3, 'fields_matched_ground_truth': 3, 'field_match_rate': 1.0}
Wrote: /Users/adi/Documents/ai_credit_intelligence/reports/document_intelligence/document_intelligence_report.md
Wrote: /Users/adi/Documents/ai_credit_intelligence/reports/document_intelligence/extraction_results.json
Wrote: /Users/adi/Documents/ai_credit_intelligence/reports/document_intelligence/extraction_results.csv



,document,field,display_name,extracted_thousands_egp,extracted_egp,ground_truth_thousands_egp,ground_truth_egp,absolute_error_thousands_egp,match,source_line,matched_label
0,english,total_revenue_2024,Total Revenue 2024,487350,487350000,487350,487350000,0,True,"Total Revenue 487,350 412,890",Total Revenue
1,english,net_income_2024,Net Income 2024,64530,64530000,64530,64530000,0,True,"Net Income (thousands EGP) 64,530 53,860",Net Income
2,english,total_assets_2024,Total Assets 2024,721350,721350000,721350,721350000,0,True,"Total Assets 721,350 634,130",Total Assets
3,arabic,total_revenue_2024,Total Revenue 2024,487350,487350000,487350,487350000,0,True,"إجمالي الإيرادات ‎E,8: EAV,F0-‏",إجمالي الإيرادات
4,arabic,net_income_2024,Net Income 2024,64530,64530000,64530,64530000,0,True,"صافي الدخل ‎OF,81: 1€,0F«‏",صافي الدخل
5,arabic,total_assets_2024,Total Assets 2024,721350,721350000,721350,721350000,0,True,"إجمالي الأصول - ‎1FE,1r- VY1,¥30‏",إجمالي الأصول


## 3. English PDF Extraction

Required values from the English sample PDF:

- Total Revenue 2024
- Net Income 2024
- Total Assets 2024

The document reports values in **thousands of EGP**, so the notebook also shows the equivalent full EGP amount.


In [4]:
english_results = extraction_df[extraction_df['document'].eq('english')].copy()
english_results[[
    'display_name',
    'extracted_thousands_egp',
    'extracted_egp',
    'ground_truth_thousands_egp',
    'absolute_error_thousands_egp',
    'match',
    'source_line',
]]


,display_name,extracted_thousands_egp,extracted_egp,ground_truth_thousands_egp,absolute_error_thousands_egp,match,source_line
0,Total Revenue 2024,487350,487350000,487350,0,True,"Total Revenue 487,350 412,890"
1,Net Income 2024,64530,64530000,64530,0,True,"Net Income (thousands EGP) 64,530 53,860"
2,Total Assets 2024,721350,721350000,721350,0,True,"Total Assets 721,350 634,130"


In [5]:
english_accuracy = results['english']['accuracy']
pd.DataFrame([english_accuracy])


,target_fields,fields_extracted,fields_matched_ground_truth,field_match_rate
0,3,3,3,1.0000


**English result:** all three requested values are extracted exactly against ground truth. This is the clean case: English labels are readable, numeric strings are preserved, and the relevant statement rows remain in a usable line order.


## 4. Arabic PDF Extraction

The Arabic PDF is the same report, but it is harder for local OCR because it combines Arabic labels, right-to-left text flow, table columns, Arabic/Persian digit forms, and OCR-confusable glyphs.

After installing `tesseract-lang`, the local environment has Arabic OCR support. The extraction can recover the three sample values, but the raw OCR text shows why this is fragile.


In [6]:
arabic_results = extraction_df[extraction_df['document'].eq('arabic')].copy()
arabic_results[[
    'display_name',
    'extracted_thousands_egp',
    'extracted_egp',
    'ground_truth_thousands_egp',
    'absolute_error_thousands_egp',
    'match',
    'source_line',
]]


,display_name,extracted_thousands_egp,extracted_egp,ground_truth_thousands_egp,absolute_error_thousands_egp,match,source_line
3,Total Revenue 2024,487350,487350000,487350,0,True,"إجمالي الإيرادات ‎E,8: EAV,F0-‏"
4,Net Income 2024,64530,64530000,64530,0,True,"صافي الدخل ‎OF,81: 1€,0F«‏"
5,Total Assets 2024,721350,721350000,721350,0,True,"إجمالي الأصول - ‎1FE,1r- VY1,¥30‏"


In [7]:
arabic_accuracy = results['arabic']['accuracy']
pd.DataFrame([arabic_accuracy])


,target_fields,fields_extracted,fields_matched_ground_truth,field_match_rate
0,3,3,3,1.0000


In [8]:
comparison = extraction_df.pivot_table(
    index='display_name',
    columns='document',
    values=['extracted_thousands_egp', 'match'],
    aggfunc='first',
)
comparison


extracted_thousands_egp          match        
document                            arabic english arabic english
display_name                                                     
Net Income 2024                      64530   64530   True    True
Total Assets 2024                   721350  721350   True    True
Total Revenue 2024                  487350  487350   True    True

### Where Arabic Extraction Is Still Fragile

Arabic extraction now matches all three ground-truth values on this sample, but it should **not** be treated as production reliable without validation.

The failure point is not only label recognition. The Arabic labels are mostly readable after installing Arabic traineddata, but numeric cells are still distorted by OCR. The extraction succeeds here because the prototype normalizes predictable OCR confusions in these sample tables.


In [9]:
arabic_ocr_text = (REPORT_DIR / 'arabic_ocr_text.txt').read_text(encoding='utf-8')
interesting_lines = []
for line_number, line in enumerate(arabic_ocr_text.splitlines(), start=1):
    if any(term in line for term in ['إجمالي الإيرادات', 'صافي الدخل', 'إجمالي الأصول']):
        interesting_lines.append({'line_number': line_number, 'raw_ocr_line': line})

pd.DataFrame(interesting_lines)


,line_number,raw_ocr_line
0,55,"على صعيد الميزانية العمومية, بلغت إجمالي الأصول ‎۷١٠,۳٠١‏ ألف جنيه مع تحسن ملحوظ في رأس"
1,85,"صافي الدخل (ألف ج.م) -1£,0 ‎OF,۸1-‏"
2,114,"إجمالي الإيرادات ‎E,۸: EAV,F0-‏"
3,123,"صافي الدخل ‎OF,۸1: 1€,0F«‏"
4,140,"‏إجمالي الأصول المتداولة - ‎FI, EPI,AE‏"
5,142,"إجمالي الأصول - ‎1FE,1r- VY1,¥۳0‏"
6,158,"صافي الدخل قبل الضرائب ‎۸۳,۲٤۰١‏"
7,222,يمثل أكبر خمسة عملاء ما نسبته 1۸ من إجمالي الإيرادات. يمثل العميل الأكبر منفردا


Examples of raw Arabic OCR distortion visible above:

- `487,350` appears as something like `EAV,F0-`.
- `64,530` appears as something like `1€,0F«`.
- `721,350` appears as something like `VY1,¥30`.

That is why a production system needs OCR confidence, table-layout parsing, accounting consistency checks, and exception routing. A brittle regex alone is not enough.


## 5. API Comparison

The comparison below uses the actual local Tesseract run plus public pricing assumptions for commercial OCR APIs.

References:

- AWS Textract pricing: https://aws.amazon.com/textract/pricing/
- Google Document AI pricing: https://cloud.google.com/document-ai/pricing
- Azure AI Document Intelligence pricing: https://azure.microsoft.com/pricing/details/ai-document-intelligence/


In [10]:
pages_per_document = results['english']['page_count']
volumes = pd.DataFrame({
    'monthly_documents': [50, 500],
})
volumes['monthly_pages'] = volumes['monthly_documents'] * pages_per_document

pricing = pd.DataFrame([
    {
        'approach': 'Local Tesseract OCR',
        'pricing_basis': 'Open-source software; local compute/ops cost only',
        'price_per_page_usd': 0.0,
        'price_per_12_page_document_usd': 0.0,
        'expected_accuracy_tradeoff': 'Worked on this sample after Arabic OCR plus custom normalization; lower robustness across new statement layouts.',
    },
    {
        'approach': 'AWS Textract Detect Document Text',
        'pricing_basis': 'Public OCR text detection price, commonly quoted at $0.0015/page for first 1M pages in US regions',
        'price_per_page_usd': 0.0015,
        'price_per_12_page_document_usd': pages_per_document * 0.0015,
        'expected_accuracy_tradeoff': 'Managed OCR with better scan handling than local setup; still needs table/field validation for financial statements.',
    },
    {
        'approach': 'Google Document AI Enterprise OCR',
        'pricing_basis': 'Public OCR processor pricing, $1.50 per 1,000 pages',
        'price_per_page_usd': 1.50 / 1000,
        'price_per_12_page_document_usd': pages_per_document * 1.50 / 1000,
        'expected_accuracy_tradeoff': 'Managed OCR/layout platform; likely better Arabic/table handling than local Tesseract, but field-level validation remains required.',
    },
    {
        'approach': 'Azure AI Document Intelligence Read/Layout',
        'pricing_basis': 'Managed document OCR/layout API; use current Azure regional pricing before production commitment',
        'price_per_page_usd': None,
        'price_per_12_page_document_usd': None,
        'expected_accuracy_tradeoff': 'Strong managed option with layout extraction; final cost and Arabic table accuracy should be benchmarked on Rawasi documents.',
    },
])

pricing


,approach,pricing_basis,price_per_page_usd,price_per_12_page_document_usd,expected_accuracy_tradeoff
0,Local Tesseract OCR,Open-source software; local compute/ops cost only,0.0000,0.0000,Worked on this sample after Arabic OCR plus custom normalization; lower robustness across new statement layouts.
1,AWS Textract Detect Document Text,"Public OCR text detection price, commonly quoted at $0.0015/page for first 1M pages in US regions",0.0015,0.0180,Managed OCR with better scan handling than local setup; still needs table/field validation for financial statements.
2,Google Document AI Enterprise OCR,"Public OCR processor pricing, $1.50 per 1,000 pages",0.0015,0.0180,"Managed OCR/layout platform; likely better Arabic/table handling than local Tesseract, but field-level validation remains required."
3,Azure AI Document Intelligence Read/Layout,Managed document OCR/layout API; use current Azure regional pricing before production commitment,NaN,NaN,Strong managed option with layout extraction; final cost and Arabic table accuracy should be benchmarked on Rawasi documents.


In [11]:
cost_rows = []
for _, price_row in pricing.dropna(subset=['price_per_page_usd']).iterrows():
    for _, volume_row in volumes.iterrows():
        cost_rows.append({
            'approach': price_row['approach'],
            'monthly_documents': int(volume_row['monthly_documents']),
            'monthly_pages': int(volume_row['monthly_pages']),
            'estimated_monthly_ocr_cost_usd': volume_row['monthly_pages'] * price_row['price_per_page_usd'],
            'estimated_cost_per_document_usd': price_row['price_per_12_page_document_usd'],
        })

pd.DataFrame(cost_rows)


,approach,monthly_documents,monthly_pages,estimated_monthly_ocr_cost_usd,estimated_cost_per_document_usd
0,Local Tesseract OCR,50,600,0.0000,0.0000
1,Local Tesseract OCR,500,6000,0.0000,0.0000
2,AWS Textract Detect Document Text,50,600,0.9000,0.0180
3,AWS Textract Detect Document Text,500,6000,9.0000,0.0180
4,Google Document AI Enterprise OCR,50,600,0.9000,0.0180
5,Google Document AI Enterprise OCR,500,6000,9.0000,0.0180


## 6. Recommendation

**Pilot phase: fewer than 50 documents/month**

Use the local Tesseract prototype as the first pilot baseline, but only with strict validation and human review for exceptions. It has zero software unit cost and now extracts both the English and Arabic sample values exactly. That said, the Arabic success depends on custom normalization of distorted OCR strings, so I would not let it auto-populate credit-model inputs without validation. At 50 documents/month and 12 pages per document, a managed OCR API such as Google Document AI or AWS Textract would cost roughly **$0.90/month** for OCR-only page processing at $0.0015/page. The cost is negligible; the real pilot decision is speed and reliability. If the pilot must support varied Arabic scans immediately, use a managed OCR API as the primary extractor and keep Tesseract as a regression baseline.

**Scale phase: around 500 documents/month**

Move to a managed OCR/layout API as the primary pipeline, with Tesseract retained for offline regression testing and cost-control experiments. At 500 documents/month and 12 pages per document, OCR-only page processing at $0.0015/page is about **$9/month** before any premium layout/model charges. The API cost is still small compared with the operational cost of bad extraction. The scale architecture should store raw OCR text, page images, extracted fields, confidence scores, source snippets, and validation outcomes.

**Expected accuracy tradeoff**

Tesseract is attractive for transparency and zero licensing cost, and it can work on controlled PDFs. Its weakness is robustness: Arabic table digits and right-to-left layouts require custom correction logic. Managed APIs should improve layout handling, confidence scoring, and scan tolerance, but they still need validation because financial-statement extraction errors can directly affect hard filters and credit features.

**Validation gates before feeding Layer 1**

Use automatic checks before the extracted numbers become model inputs:

- Require a source line and confidence/normalization trace for every extracted value.
- Check accounting identities, for example total assets equals liabilities plus equity when those fields are available.
- Compare extracted revenue, income, and assets against prior-year values for extreme jumps.
- Route low-confidence, missing, or inconsistent values to human review.
- Keep both raw OCR text and normalized values for auditability.


## 7. Final Answer Summary

| Requirement | Answer |
| --- | --- |
| English extraction | 3/3 exact: revenue 487,350; net income 64,530; total assets 721,350, all in thousands EGP. |
| Arabic extraction | 3/3 exact after installing Arabic traineddata and applying OCR-confusable numeric normalization. |
| Arabic accuracy difference | English is clean label/number extraction. Arabic requires layout-aware OCR and normalization because raw numbers are distorted. |
| Open-source approach | Local Tesseract with ImageMagick rendering. Good prototype, weak production robustness. |
| Commercial approach | Google Document AI or AWS Textract. Low OCR unit cost and stronger production controls; benchmark Arabic financial statements before rollout. |
| Pilot recommendation | Tesseract baseline plus validation; managed OCR primary if document variety is high. |
| Scale recommendation | Managed OCR/layout API with validation gates, audit storage, and exception review. |
